# CRTT — SP500 Structural Transition Analysis

**Repository:** [Motor-de-Velos-SCM](https://github.com/sergiocamaramadrid-cyber/Motor-de-Velos-SCM)  
**Version:** v2.6-experimental  
**Status:** Experimental — reproducible and validated.

---

This notebook applies CRTT (Critical Regime Transition Test) to SP500-like financial time-series data
to demonstrate detection of a strong structural transition (`foreground_confirmed`).

The SP500 dataset serves as the canonical **foreground** reference in the SCM-RAA validation suite.

## 0. Setup

In [ ]:
# Install dependencies (Colab)
# !pip install numpy pandas scipy matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print('Setup complete.')

## 1. CRTT Core

In [ ]:
def fit_linear(x, y):
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    rss = np.sum((y - y_hat) ** 2)
    sigma2 = rss / n
    log_lik = -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)
    aic = 2 * 3 - 2 * log_lik
    return aic, beta, y_hat


def fit_piecewise(x, y, threshold):
    n = len(x)
    mask = x < threshold
    aic_total, betas = 0.0, []
    for seg in [mask, ~mask]:
        if seg.sum() < 3:
            return np.inf, None
        a, b, _ = fit_linear(x[seg], y[seg])
        aic_total += a
        betas.append(b)
    return aic_total, betas


def crtt_scan(x, y, n_thresholds=100):
    x, y = np.asarray(x, float), np.asarray(y, float)
    aic_lin, _, y_hat_lin = fit_linear(x, y)
    lo, hi = np.percentile(x, 10), np.percentile(x, 90)
    thresholds = np.linspace(lo, hi, n_thresholds)
    aic_pw_scan = []
    for thr in thresholds:
        a, _ = fit_piecewise(x, y, thr)
        aic_pw_scan.append(a)
    aic_pw_scan = np.array(aic_pw_scan)
    best_idx = np.argmin(aic_pw_scan)
    best_thr = thresholds[best_idx]
    best_aic_pw = aic_pw_scan[best_idx]
    delta_aic = aic_lin - best_aic_pw
    return {
        'delta_aic': delta_aic,
        'threshold': best_thr,
        'aic_linear': aic_lin,
        'aic_piecewise': best_aic_pw,
        'thresholds': thresholds,
        'aic_pw_scan': aic_pw_scan,
        'x': x, 'y': y,
        'y_hat_linear': y_hat_lin,
        'n': len(x),
    }

print('CRTT scan loaded.')

## 2. SP500 Synthetic Reference Dataset

A piecewise-linear synthetic series that mimics the structural break pattern
observed in the SP500 volatility-return relationship used in the SCM-RAA validation.

In [ ]:
N = 200
x = np.linspace(0, 10, N)
# Piecewise: clear regime change at x ~ 5
y = np.where(
    x < 5,
    1.8 * x + rng.normal(0, 0.4, N),
    -0.7 * (x - 5) + 9.0 + rng.normal(0, 0.4, N)
)

plt.figure(figsize=(8, 4))
plt.scatter(x, y, s=10, alpha=0.6, label='SP500 (synthetic reference)')
plt.axvline(5.0, color='red', linestyle='--', alpha=0.7, label='True threshold')
plt.xlabel('x'); plt.ylabel('y')
plt.title('SP500 Reference Dataset — Structural Break at x=5')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Dataset: N={N}')

## 3. Run CRTT Scan

In [ ]:
result = crtt_scan(x, y)
print(f"ΔAIC = {result['delta_aic']:.2f}  (positive → piecewise better)")
print(f"Optimal threshold = {result['threshold']:.3f}")
print(f"AIC linear        = {result['aic_linear']:.2f}")
print(f"AIC piecewise     = {result['aic_piecewise']:.2f}")

## 4. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: AIC scan
ax = axes[0]
ax.plot(result['thresholds'], result['aic_pw_scan'], color='steelblue', label='AIC piecewise')
ax.axhline(result['aic_linear'], color='orange', linestyle='--', label='AIC linear')
ax.axvline(result['threshold'], color='red', linestyle=':', label=f"Best thr={result['threshold']:.2f}")
ax.set_xlabel('Threshold'); ax.set_ylabel('AIC')
ax.set_title(f'CRTT Scan — ΔAIC={result["delta_aic"]:.1f}')
ax.legend(fontsize=8)

# Right: Data + piecewise fit
ax = axes[1]
thr = result['threshold']
mask = x < thr
for seg, color in [(mask, 'steelblue'), (~mask, 'darkorange')]:
    xs, ys = x[seg], y[seg]
    ax.scatter(xs, ys, s=8, alpha=0.5, color=color)
    if seg.sum() >= 2:
        _, _, yhat = fit_linear(xs, ys)
        order = np.argsort(xs)
        ax.plot(xs[order], yhat[order], color=color, linewidth=2)
ax.axvline(thr, color='red', linestyle='--', alpha=0.7, label=f'Detected thr={thr:.2f}')
ax.plot(x, result['y_hat_linear'], 'k--', alpha=0.4, label='Linear fit')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('SP500 — Piecewise vs Linear')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. RAA Classification

In [ ]:
DELTA_AIC_STRONG = 6.0
DELTA_AIC_WEAK   = 2.0
STRONG_RATE_THRESHOLD  = 0.6
FAILURE_RATE_THRESHOLD = 0.6

da = result['delta_aic']
if da >= DELTA_AIC_STRONG:
    status = 'strong'
elif da >= DELTA_AIC_WEAK:
    status = 'weak'
else:
    status = 'failure'

layer = {'strong': 'foreground', 'weak': 'midground', 'failure': 'background'}[status]
print(f'RAA → status={status}, layer={layer}')

# Bootstrap
statuses = []
rng_b = np.random.default_rng(RANDOM_SEED)
for _ in range(300):
    idx = rng_b.integers(0, N, N)
    try:
        r = crtt_scan(x[idx], y[idx], n_thresholds=50)
        sda = r['delta_aic']
        statuses.append('strong' if sda >= DELTA_AIC_STRONG else ('weak' if sda >= DELTA_AIC_WEAK else 'failure'))
    except Exception:
        statuses.append('failure')

strong_rate  = statuses.count('strong')  / len(statuses)
failure_rate = statuses.count('failure') / len(statuses)
weak_rate    = statuses.count('weak')    / len(statuses)
print(f'Bootstrap (n=300): strong_rate={strong_rate:.2f}, weak_rate={weak_rate:.2f}, failure_rate={failure_rate:.2f}')

if status == 'strong' and strong_rate >= STRONG_RATE_THRESHOLD:
    verdict = 'foreground_confirmed'
elif failure_rate >= FAILURE_RATE_THRESHOLD:
    verdict = 'background_confirmed'
else:
    verdict = 'midground_candidate'

print(f'\n✔ VERDICT: {verdict}')

## 6. Summary

In [ ]:
summary = pd.DataFrame([{
    'dataset': 'SP500',
    'n': result['n'],
    'delta_aic': round(result['delta_aic'], 2),
    'threshold': round(result['threshold'], 3),
    'status': status,
    'layer': layer,
    'strong_rate': round(strong_rate, 3),
    'weak_rate': round(weak_rate, 3),
    'failure_rate': round(failure_rate, 3),
    'verdict': verdict,
}])
summary